# Phase 0 — Build Musketeer Stockfish (compile loop)

This notebook compiles the **Musketeer Stockfish** source on Google Colab
(which has a C++ toolchain), establishing the build loop needed for the NNUE
integration. It builds the engine **unchanged** first, to confirm a clean
baseline before any NNUE code is added.

**You need:** the source archive `Musketeer-Stockfish-master 17-07-26.zip`
(uploaded in Step 2).

A GPU runtime is **not** required for building — a normal CPU runtime is fine.

## Step 1 — Toolchain check (Colab already has g++ and make)

In [ ]:
!g++ --version | head -1
!make --version | head -1
# CPU features decide the ARCH we can build (bmi2/avx2):
!grep -o -m1 -E 'bmi2|avx2|sse4_1|popcnt' /proc/cpuinfo | sort -u | tr '\n' ' '; print()

## Step 2 — Upload and unpack the engine source
Run the cell, then choose `Musketeer-Stockfish-master 17-07-26.zip`.

In [ ]:
import os, zipfile, glob
from google.colab import files
up = files.upload()                       # pick the source zip
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/msk')
# locate the src directory (contains the Makefile)
src = os.path.dirname(glob.glob('/content/msk/**/Makefile', recursive=True)[0])
os.chdir(src)
print('source dir:', src)
print('files:', sorted(os.listdir('.'))[:12], '...')

## Step 3 — Compile
Uses `x86-64-modern` (portable, needs only AVX2/POPCNT). If Step 1 showed
`bmi2`, you may switch to `ARCH=x86-64-bmi2` for a slightly faster binary.

In [ ]:
!make clean >/dev/null 2>&1; echo 'building...'
!make build ARCH=x86-64-modern -j$(nproc) 2>&1 | tail -25

## Step 4 — Verify the compiled engine runs and plays Musketeer

In [ ]:
import glob, subprocess
exe = sorted(glob.glob('musketeer-stockfish*') + glob.glob('stockfish*'))
exe = [e for e in exe if os.access(e, os.X_OK) and not e.endswith('.o')][0]
print('built engine:', exe)

def run(cmds):
    p = subprocess.run(['./'+exe], input='\n'.join(cmds)+'\n',
                       capture_output=True, text=True, timeout=60)
    return p.stdout

print(run(['uci', 'quit']).splitlines()[0])       # id name ...
start = '*u***h**/rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR/HU****** w KQkq - 0 1'
out = run(['setoption name UCI_Variant value musketeer',
           f'position fen {start}', 'go depth 10', 'quit'])
print([l for l in out.splitlines() if l.startswith('bestmove')][-1])

## Step 5 — Download the compiled Linux engine
This baseline binary confirms the build loop works. The NNUE integration
(adding the `nnue/` sources, `EvalFile`, and the feature set) then proceeds
against this same build, per `docs/NNUE_Integration_Plan.md`.

In [ ]:
from google.colab import files
files.download(exe)